# Session 1, Live Demo: Camac Drinks' advertising, explored *with Claude*

**This is the instructor's demo notebook. Drive it conversationally.**

Today is not a regression lesson (that's Session 5). The demo models two habits:
1. Explore a dataset by **talking to Claude**, one sharp question at a time.
2. **Catch Claude when it's wrong**. That is the core skill of this module.

> **The data:** Camac Drinks, a fictional sparkling-water brand, advertises in 200 towns across Ireland and Britain. For each town: spend on `TV`, `Radio` and `Newspaper` (in thousands of euros) and `Sales` (in thousands of cans).
>
> *Where it really comes from:* a teaching dataset, a variant of the Advertising data in *An Introduction to Statistical Learning*. The spend columns match the book; the sales figures don't. Trust this file over remembered textbook numbers.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from a public link, so there is nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session1/data/advertising.csv"

import pandas as pd

if DATA_URL:
    ads = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload advertising.csv ...")
        up = files.upload()
        ads = pd.read_csv(next(iter(up)))
    except ImportError:
        ads = pd.read_csv("advertising.csv")   # local fallback (course folder: data/advertising.csv)

ads.head()

## Beat 1: Meet the data before you analyse it

The first question to Claude is never "build me a model". It's "what am I looking at?"
Ask Claude what the **units** are. They are not in the file.

In [ ]:
ads.shape          # how many rows and columns?
ads.describe()      # ranges and averages. Does anything look odd?

Look at `Newspaper` in the table: the **mean** (about 30.6) sits above the **median** (the 50% row, about 25.8). A few towns spend heavily and drag the mean up. That is the skew from the measures-of-centre slides, in a second dataset.

## Beat 2: Predict before you compute  🎯

**Ask the room, hands up, before running anything:** which channel drives sales most: TV, Radio or Newspaper? Get them to commit. Then look.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for a, col in zip(ax, ['TV', 'Radio', 'Newspaper']):
    a.scatter(ads[col], ads['Sales'], alpha=0.6)
    a.set_xlabel(col + ' spend (€000s)')
ax[0].set_ylabel('Sales (000s of cans)')
plt.tight_layout(); plt.show()

## Beat 3: Plant the mistake  ⚠️

Invite Claude to over-claim, then catch it. Ask:

> *"Newspaper spend is positively correlated with sales, so newspaper ads drive sales, right? Should we keep the newspaper budget?"*

Then check the correlations yourself:

In [ ]:
ads.corr(numeric_only=True).round(2)

`Newspaper`'s link to `Sales` is weak (0.16), and `Newspaper` moves with `Radio` (0.35). Towns that spend on radio also spend on newspapers, so newspaper can look useful by riding along with radio. Correlation isn't causation.

Second trap: ask Claude for radio's regression coefficient **without running code**, then check it against the cell below. Make it show its work.

## Beat 4: The CFO's question

> *"The CFO wants to cut €100k of advertising. Which channel goes, and what do we lose?"*

Ask Claude for the calculation, not just the answer. A regression with all three channels holds the other two fixed. You don't need to understand regression yet (Session 5); read the coefficients as *extra thousands of cans per extra €1k of spend*.

In [ ]:
import numpy as np
X = np.column_stack([np.ones(len(ads)), ads[['TV', 'Radio', 'Newspaper']]])
coef, *_ = np.linalg.lstsq(X, ads['Sales'], rcond=None)
per_1k = pd.Series(coef[1:], index=['TV', 'Radio', 'Newspaper'])
print('Extra sales (000s of cans) per extra €1k, others held fixed:')
print(per_1k.round(4))
print()
print('Cans lost if we cut €100k from one channel:')
print((per_1k * 100 * 1000).round(-1).astype(int))

Newspaper's effect is about zero once TV and radio are in the model; cutting €100k there costs about 30 cans. Radio returns about twice what TV does per euro at the margin (≈ 107 vs ≈ 54 cans per €1k).

**Push back before you tell the CFO:** the data is observational (managers chose each town's spend), a straight-line model may not hold far outside the spend we've seen, and towns are not the whole market. How confident should the CFO be?

## Weak prompt vs strong prompt

| Weak prompt | Strong prompt |
|---|---|
| "analyse this data" | "which channel gives the most extra *sales per euro spent*? Show the calculation" |
| "make a chart" | "plot sales against each channel. Which relationship looks strongest, and why?" |
| "is this good?" | "how confident should I be, and what would make you wrong?" |

The precise, business-framed question earns the useful answer.

## Wrap

Analytics is a conversation you direct and check. Everything Claude said today, we checked against the data. Claude is a tool, not an oracle.